# Lab 1 - Exercise 2: Pipeline Consolidation

Questo notebook continua direttamente dall'Exercise 1.3. Nel notebook precedente abbiamo verificato che una ResNet-18 con backbone congelato e sola testa finale addestrabile e' una baseline utile, ma non sufficiente a superare la baseline SVM.

Qui l'obiettivo cambia: non vogliamo accumulare nuove celle di training ripetitive, ma consolidare una pipeline riproducibile per lanciare, registrare e confrontare esperimenti.

---
---
## Exercise 2: Pipeline Consolidation

Consolidate your implementation. When building applications based on Deep Learning, you will inevitably need to run many experiments. So, it is always a good idea to engineer a reproducible pipeline that allows you to run and re-run experiments with different hyperparameters.

Aspetti richiesti:

- **Model and Loss abstraction**: la pipeline deve poter cambiare modello, loss e optimizer senza riscrivere il training loop.
- **Configuration management**: gli iperparametri non devono essere sparsi nelle celle, ma raccolti in una configurazione.
- **Logging**: le run devono produrre metriche salvabili e confrontabili; WandB puo' essere usato, ma resta opzionale.

## Collegamento con Exercise 1.3

Il notebook precedente prepara una baseline head-only: ResNet-18 pre-addestrata, backbone congelato e nuova testa finale per le 43 classi GTSRB.

Exercise 2 serve a trasformare questa logica in una pipeline riproducibile. Quando cambiano split, batch size, optimizer o loss, dobbiamo poter ricostruire chiaramente cosa e' stato eseguito e confrontare i risultati prodotti dalla pipeline corrente.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dla_lab1.config import experiment_config, load_config
from dla_lab1.experiments import batch_size_for, run_finetuning
from dla_lab1.tracking import (
    experiment_output_dir,
    save_run_artifacts,
    summarize_history,
    wandb_is_available,
)

config = load_config(ROOT / "config" / "config.yaml")

# Lasciato False per evitare di rilanciare training lunghi quando si esegue il notebook.
RUN_TRAINING = False

# WandB e' opzionale. Attivalo solo dopo aver installato wandb e fatto login.
ENABLE_WANDB = True
config["wandb"]["enabled"] = ENABLE_WANDB
config["wandb"]["group"] = "exercise_2_pipeline_consolidation"

print(f"Project root: {ROOT}")
print(f"WandB installed: {wandb_is_available()}")
print(f"WandB enabled for this notebook: {config['wandb']['enabled']}")

## Cosa era stato fatto nella versione esplorativa

Nella versione esplorativa l'idea principale era corretta: era stata creata una funzione `train_and_evaluate` e poi erano stati previsti diversi esperimenti cambiando loss e optimizer.

Il limite era la forma: molte configurazioni erano scritte direttamente nelle celle, alcune celle erano quasi duplicate e WandB era gestito manualmente nel notebook. Nel refactoring, questa logica viene spostata in `src/dla_lab1` e controllata da `config/config.yaml`.


In [ ]:
planned_head_only_screening = pd.DataFrame([
    ["Adam_CrossEntropy", "CrossEntropy", "Adam", "baseline head-only principale"],
    ["AdamW_CrossEntropy", "CrossEntropy", "AdamW", "alternativa con AdamW"],
    ["SGD_CrossEntropy", "CrossEntropy", "SGD", "optimizer classico con momentum"],
    ["Adam_WeightedCrossEntropy", "WeightedCrossEntropy", "Adam", "class imbalance"],
    ["AdamW_WeightedCrossEntropy", "WeightedCrossEntropy", "AdamW", "class imbalance + AdamW"],
    ["SGD_WeightedCrossEntropy", "WeightedCrossEntropy", "SGD", "class imbalance + SGD"],
    ["Adam_FocalLoss", "FocalLoss", "Adam", "classi difficili"],
    ["AdamW_FocalLoss", "FocalLoss", "AdamW", "classi difficili + AdamW"],
    ["SGD_FocalLoss", "FocalLoss", "SGD", "classi difficili + SGD"],
], columns=["Experiment", "Loss", "Optimizer", "Reason"])

planned_head_only_screening


### Commento sullo screening

Questa tabella non riporta risultati: e' solo l'elenco delle prove che la pipeline permette di eseguire in modo ordinato.

La scelta di non duplicare tutte le run nel notebook serve a mantenere il lavoro leggibile. Se una variante viene rilanciata, il risultato deve essere salvato dalla pipeline corrente in `artifacts/runs` oppure tracciato su WandB.


## Nuova architettura della pipeline

La pipeline e' divisa in moduli piccoli. Ogni funzione importante e' commentata nel relativo file `.py`, cosi' il notebook resta leggibile e non diventa un blocco di implementazione.

In [ ]:
pipeline_components = pd.DataFrame([
    ["config.py", "load_config, experiment_config", "Legge il YAML e costruisce la configurazione finale della run."],
    ["data.py", "build_dataloaders", "Crea train/validation/test loader e split anti-leakage."],
    ["models.py", "build_classifier", "Crea ResNet-18 e sostituisce la testa finale per 43 classi."],
    ["losses.py", "build_loss", "Seleziona CrossEntropy, WeightedCrossEntropy o FocalLoss."],
    ["train.py", "build_optimizer, train_model", "Crea optimizer/scheduler e gestisce training, validation, checkpoint."],
    ["evaluate.py", "predict, classification_metrics", "Calcola predizioni e metriche di classificazione."],
    ["tracking.py", "save_run_artifacts", "Salva history, config e summary della run in artifacts/runs."],
], columns=["File", "Funzioni principali", "Utilizzo"])

pipeline_components

## Configuration management

Gli iperparametri principali sono centralizzati in `config/config.yaml`.

Questo rende piu' semplice:

- cambiare batch size;
- cambiare optimizer;
- cambiare loss;
- abilitare o disabilitare WandB;
- rilanciare una run con gli stessi parametri.

In [ ]:
available_experiments = pd.DataFrame([
    [name, exp.get("description", ""), exp.get("batch_size_key", "")]
    for name, exp in config["experiments"].items()
], columns=["experiment_name", "description", "batch_size_key"])

available_experiments

In [ ]:
selected_experiment = "ex1_3_head_only_adam_ce"
exp_cfg = experiment_config(config, selected_experiment)
batch_size = batch_size_for(config, exp_cfg["experiment"]["batch_size_key"])

selected_config_summary = pd.DataFrame([
    ["experiment", selected_experiment],
    ["model", exp_cfg["model"]["name"]],
    ["freeze_backbone", exp_cfg["model"]["freeze_backbone"]],
    ["unfreeze_layer4", exp_cfg["model"].get("unfreeze_layer4", False)],
    ["loss", exp_cfg["training"]["loss"]],
    ["optimizer", exp_cfg["training"]["optimizer"]],
    ["learning_rate", exp_cfg["training"]["learning_rate"]],
    ["batch_size", batch_size],
    ["epochs", exp_cfg["training"]["epochs"]],
    ["scheduler", exp_cfg["training"]["scheduler"]],
], columns=["parameter", "value"])

selected_config_summary

## Logging locale e WandB

Il logging locale e' sempre disponibile e salva i risultati in:

`artifacts/runs/<experiment_name>/`

Dentro questa cartella vengono salvati:

- `history.csv`: loss, accuracy e learning rate per epoca;
- `config_used.yaml`: configurazione usata nella run;
- `summary.json`: riassunto con best epoch e best validation accuracy.

WandB e' supportato dalla pipeline, ma non e' obbligatorio. Per usarlo bisogna installare `wandb`, fare login e impostare `ENABLE_WANDB = True`.

In [ ]:
wandb_status = pd.DataFrame([
    ["wandb_installed", wandb_is_available()],
    ["wandb_enabled", config["wandb"]["enabled"]],
    ["wandb_project", config["wandb"]["project"]],
    ["wandb_group", config["wandb"]["group"]],
    ["local_output_dir", experiment_output_dir(config, selected_experiment, root=ROOT)],
], columns=["item", "value"])

wandb_status

## Setup riproducibile dell'Exercise 1.3

Qui mostriamo quale configurazione viene passata alla pipeline per rilanciare la baseline head-only dell'Exercise 1.3.


In [ ]:
exercise_13_setup = pd.DataFrame([
    ["experiment", selected_experiment],
    ["model", exp_cfg["model"]["name"]],
    ["freeze_backbone", exp_cfg["model"].get("freeze_backbone")],
    ["unfreeze_layer4", exp_cfg["model"].get("unfreeze_layer4", False)],
    ["loss", exp_cfg["training"]["loss"]],
    ["optimizer", exp_cfg["training"]["optimizer"]],
    ["learning_rate", exp_cfg["training"]["learning_rate"]],
    ["epochs", exp_cfg["training"]["epochs"]],
    ["batch_size", batch_size],
], columns=["parameter", "value"])

exercise_13_setup


In [ ]:
artifact_dir = experiment_output_dir(config, selected_experiment, root=ROOT)
artifact_check = pd.DataFrame([
    ["expected_output_dir", artifact_dir],
    ["history_csv_exists", (artifact_dir / "history.csv").exists()],
    ["summary_json_exists", (artifact_dir / "summary.json").exists()],
    ["config_snapshot_exists", (artifact_dir / "config_used.yaml").exists()],
], columns=["item", "value"])

artifact_check


### Commento sul setup

La pipeline e' pronta per produrre risultati confrontabili. Dopo aver rilanciato un esperimento, la history viene salvata in history.csv e il riepilogo in summary.json dentro la cartella della run.


## Cella opzionale per rilanciare una run

La cella seguente e' disattivata di default. Per usarla:

1. imposta `RUN_TRAINING = True`;
2. se vuoi WandB, installa `wandb`, fai login e imposta `ENABLE_WANDB = True`;
3. scegli `experiment_to_run`.

La funzione `run_finetuning` salva automaticamente checkpoint e artifact locali. Se WandB e' abilitato, invia anche le metriche online nel progetto/gruppo indicato dal config.

In [ ]:
experiment_to_run = "ex1_3_head_only_adam_ce"

if RUN_TRAINING:
    config["wandb"]["enabled"] = ENABLE_WANDB
    config["wandb"]["group"] = "exercise_2_pipeline_consolidation"
    result = run_finetuning(config, experiment_to_run, root=ROOT)
    run_summary = result["artifacts"]["summary"]
    print(f"Run salvata in: {result['artifacts']['output_dir']}")
    pd.DataFrame([run_summary])
else:
    print("Training non eseguito. Imposta RUN_TRAINING = True per rilanciare la run.")

## Conclusione Exercise 2

La pipeline ora soddisfa le richieste dell'esercizio:

- il modello e' astratto in `build_classifier`;
- loss e optimizer sono selezionati da configurazione;
- gli iperparametri sono centralizzati in `config.yaml`;
- il training loop e' riutilizzabile;
- i risultati sono salvati localmente in modo riproducibile;
- WandB e' predisposto come logging opzionale, senza API key nel notebook.

Il notebook non e' piu' una sequenza di celle duplicate: diventa un'interfaccia pulita per configurare, lanciare e confrontare esperimenti.